# ភាគបន្ថែម E៖ Parameter-efficient Finetuning ជាមួយ LoRA

> **ចំណាំអំពីប្រភព និងអាជ្ញាបណ្ណ** — នេះជា *companion notebook* របស់សហគមន៍ខ្មែរ។
> កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង notebook ផ្លូវការរបស់ឃ្លាំង
> [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/appendix-E/01_main-chapter-code/appendix-E.ipynb) (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។
> ចំណែកអត្ថបទពន្យល់ជាភាសាខ្មែរគឺជាអត្ថបទដើមរបស់សហគមន៍ (CC BY 4.0) —
> **មិនមែន**ជាការបកប្រែអត្ថបទក្នុងសៀវភៅ *Build a Large Language Model (From Scratch)* ទេ
> ព្រោះអត្ថបទសៀវភៅជាកម្មសិទ្ធិរបស់ Manning Publications និងស្ថិតក្រៅវិសាលភាពអាជ្ញាបណ្ណបើកចំហ។
> សម្រាប់ការពន្យល់ពេញលេញ សូមអានសៀវភៅផ្លូវការ។

**កូដ៖** តក្កវិទ្យាកូដ (logic) ដូច upstream ទាំងស្រុង — មានតែ *comments* ក្នុងកូដដែលត្រូវបានបកប្រែជាភាសាខ្មែរដោយសហគមន៍; កូដដែល comment ជាប់ (commented-out code) នៅដដែល។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតជំពូកពាក់ព័ន្របស់ឃ្លាំង upstream
(ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការ ដូចជា `previous_chapters.py`, `gpt_download.py`
និងឯកសារទិន្នន័យ) រួចដំឡើង dependencies តាម `requirements.txt` ឬប្រើ Google Colab។

## មាតិកា

- ## ០. ការត្រៀមបរិស្ថាន
- ## ១. ការរៀបចំ DataLoader
- ## ២. ការផ្ទុកម៉ូដែល pretrained
- ## ៣. Accuracy មុន finetuning
- ## ៤. ក្លាស LinearWithLoRA
- ## ៥. ការជំនួស Linear ដោយ LoRA
- ## ៦. ការបណ្តុះបណ្តាល LoRA
- ## ៧. លទ្ធផលចុងក្រោយ

In [ ]:
# ការតភ្ជាប់ឯកសារជំនួយ — ឱ្យដំណើរការភ្ជាប់ដូច notebook ដើមរបស់ upstream
# (ចង្អុល sys.path និងថតធ្វើការទៅកាន់ថតកូដក្នុងឃ្លាំងកសារខ្មែរនេះ)
import os, sys
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "កូដ")):
    _p = os.path.dirname(_p)
_UP = os.path.join(_p, "កូដ", "appendix-E/01_main-chapter-code")
sys.path.insert(0, _UP)
os.chdir(_UP)


## ០. ការត្រៀមបរិស្ថាន

ពិនិត្យកំណែបណ្ណាល័យ និងនាំចូលឧបករណ៍។

In [ ]:
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
        "tensorflow",  # សម្រាប់ weights pretrained របស់ OpenAI
        "pandas"  # ការផ្ទុក dataset
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# import urllib
import requests
from pathlib import Path
import pandas as pd
from previous_chapters import (
    download_and_unzip_spam_data,
    create_balanced_dataset,
    random_split
)
# បើឯកសារ `previous_chapters.py` មិនមានក្នុងមូលដ្ឋានទេ អ្នកអាច import វាពី package PyPI `llms-from-scratch`។
# សម្រាប់លម្អិត សូមមើល៖ https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# ឧទាហរណ៍៖
# from llms_from_scratch.ch06 import (
#     download_and_unzip_spam_data,
#     create_balanced_dataset,
#     random_split
# )



url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"Primary URL failed: {e}. Trying backup URL...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

# សៀវភៅដើមបានប្រើ
# except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as e:
# in the code above.
# ប៉ុន្តែអ្នកប្រើ VPN ខ្លះបានរាយការណ៍ពីបញ្ហា `urllib` ដូច្នេះកូដត្រូវបានធ្វើបច្ចុប្បន្នភាពឱ្យប្រើ `requests` ជំនួសវិញ

df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
balanced_df = create_balanced_dataset(df)
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

## ១. ការរៀបចំ DataLoader

បង្កើត dataloaders សម្រាប់កិច្ចការចំណាត់ថ្នាក់។

In [ ]:
import torch
import tiktoken
from previous_chapters import SpamDataset


tokenizer = tiktoken.get_encoding("gpt2")
train_dataset = SpamDataset("train.csv", max_length=None, tokenizer=tokenizer)
val_dataset = SpamDataset("validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
test_dataset = SpamDataset("test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)

In [ ]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [ ]:
print("Train loader:")
for input_batch, target_batch in train_loader:
    pass

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

In [ ]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

## ២. ការផ្ទុកម៉ូដែល pretrained

ផ្ទុក GPT-2 pretrained និងរៀបចំ output head។

In [ ]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt
# ជាជម្រើស៖
# from llms_from_scratch.ch04 import GPTModel
# from llms_from_scratch.ch05 import load_weights_into_gpt



CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,  # ទំហំ vocabulary
    "context_length": 1024,  # ប្រវែង context
    "drop_rate": 0.0,  # អត្រា dropout
    "qkv_bias": True  # bias សម្រាប់ query-key-value
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

In [ ]:
from previous_chapters import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)


text_1 = "Every effort moves you"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

In [ ]:
torch.manual_seed(123)

num_classes = 2
model.out_head = torch.nn.Linear(in_features=768, out_features=num_classes)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # ប្រើ PyTorch 2.9 ឬថ្មីជាងនេះ សម្រាប់លទ្ធផល mps ស្ថិរ
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

model.to(device);

## ៣. Accuracy មុន finetuning

វាស់ accuracy គោលដើម្បីប្រៀបធៀប។

In [ ]:
from previous_chapters import calc_accuracy_loader
# ជាជម្រើស៖
# from llms_from_scratch.ch06 import calc_accuracy_loader



torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

In [ ]:
import math

class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))  # ស្រដៀងនឹងការចាប់ផ្តើម weights ស្តង់ដារ
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        # ចំណាំ៖ ជំពូកដើមមិនបានរួមបញ្ចូលការធ្វើមាត្រដ្ឋានដោយ self.rank ទេ
        # ការធ្វើមាត្រដឋាននេះមិនចាំបាច់ទេ ប៉ុន្តែវាស្តង់ដារ និងងាយស្រួលជាង
        # ព្រោះវាអនុញ្ញាតឱ្យប្រៀបធៀបការរត់រវាង ranks ផ្សេងៗ ដោយមិនចាំបាច់កែ learning rates
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x

## ៤. ក្លាស LinearWithLoRA

បន្ថែម low-rank adapters (A, B) ទៅលើស្រទាប់ Linear។

In [ ]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

## ៥. ការជំនួស Linear ដោយ LoRA

ជំនួសគ្រប់ស្រទាប់ Linear ហើយរាប់ trainable parameters ដែលថយចុះខ្លាំង។

In [ ]:
def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            # ជំនួសស្រទាប់ Linear ដោយ LinearWithLoRA
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            # អនុវត្តអនុគមន៍ដូចគ្នាជាបន្តបន្ទាប់ទៅម៉ូឌុលកូន
            replace_linear_with_lora(module, rank, alpha)

In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before: {total_params:,}")

for param in model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters after: {total_params:,}")

In [ ]:
replace_linear_with_lora(model, rank=16, alpha=16)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")

## ៦. ការបណ្តុះបណ្តាល LoRA

train តែ adapters ប៉ុណ្ណោះ ហើយតាមដាន loss/accuracy។

In [ ]:
model.to(device)

print(model)

In [ ]:
torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

In [ ]:
import time
from previous_chapters import train_classifier_simple
# ជាជម្រើស៖
# from llms_from_scratch.ch06 import train_classifier_simple


start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=0.1)

num_epochs = 5
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
from previous_chapters import plot_values
# ជាជម្រើស៖
# from llms_from_scratch.ch06 import plot_values

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses, label="loss")

## ៧. លទ្ធផលចុងក្រោយ

វាយតម្លៃ accuracy បន្ទាប់ពី LoRA finetuning។

In [ ]:
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ